# Kepler Exoplanet Analysis

In this project, we are analyzing the Kepler Exoplanet Search Results dataset
to understand the data, find useful patterns, check relationships between
features, and try a simple machine learning model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

project_root = Path.cwd()

if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from src.data_loader import load_data

df = load_data()

print("Dataset shape:", df.shape)
df.head()

ModuleNotFoundError: No module named 'src'

In [ ]:
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

In [ ]:
print("Classification count:")
print(df["exoplanet_archive_disposition"].value_counts())

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(15))

print("\nBasic statistics:")
df.describe()

In [ ]:
print("\nPlanet features:")
df[
    [
        "orbital_period_days",
        "transit_depth_ppm",
        "planetary_radius_earth_radii",
        "equilibrium_temperature_k"
    ]
].describe()

In [ ]:
plt.figure(figsize=(7, 5))

df["exoplanet_archive_disposition"].value_counts().plot(kind="bar")

plt.title("Kepler Object Classification")
plt.xlabel("Classification")
plt.ylabel("Number of Objects")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    df["planetary_radius_earth_radii"],
    df["transit_depth_ppm"],
    alpha=0.5
)

plt.title("Planetary Radius vs Transit Depth")
plt.xlabel("Planetary Radius (Earth Radii)")
plt.ylabel("Transit Depth (ppm)")

plt.tight_layout()
plt.show()

In [ ]:
features = [
    "orbital_period_days",
    "impact_parameter",
    "transit_duration_hours",
    "transit_depth_ppm",
    "planetary_radius_earth_radii",
    "equilibrium_temperature_k",
    "insolation_flux",
    "transit_signal_to_noise_ratio",
]

correlation = df[features].corr()

print(correlation.round(2))

In [ ]:
from scipy.stats import pearsonr

data = df[
    ["planetary_radius_earth_radii", "transit_depth_ppm"]
].dropna()

r, p = pearsonr(
    data["planetary_radius_earth_radii"],
    data["transit_depth_ppm"]
)

print("Pearson correlation:", r)
print("p-value:", p)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

features = [
    "orbital_period_days",
    "impact_parameter",
    "transit_duration_hours",
    "transit_depth_ppm",
    "planetary_radius_earth_radii",
    "equilibrium_temperature_k",
    "insolation_flux",
    "transit_signal_to_noise_ratio",
]

target = "exoplanet_archive_disposition"

data = df[features + [target]].dropna()

X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=features
).sort_values(ascending=False)

print(importance)

### Screening

The screening score is used to find objects that the model considers
more suitable for further review. It is only a prioritization score and
does not confirm an exoplanet.

In [ ]:
probabilities = model.predict_proba(X)

results = data.copy()

for i, name in enumerate(model.classes_):
    results[f"probability_{name.lower().replace(' ', '_')}"] = probabilities[:, i]

results["screening_score"] = (
    results["probability_confirmed"] +
    results["probability_candidate"]
) * 100

results = results.sort_values(
    "screening_score",
    ascending=False
)

results.head(20)